In [15]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from numba import njit
import pickle

In [2]:
wordvec_model = Word2Vec.load("./word2vec.model")
sept = pd.read_pickle("./pickles/sept.pickle")
tisch = pd.read_pickle("./pickles/tisch.pickle")

In [3]:
tisch

,text,str,verse,chapter,book,rmac
0,βίβλος,976,1,1,40,n-nsf
1,γενέσεως,1078,1,1,40,n-gsf
2,ἰησοῦ,2424,1,1,40,n-gsm
3,χριστοῦ,5547,1,1,40,n-gsm
4,υἱοῦ,5207,1,1,40,n-gsm
...,...,...,...,...,...,...
137520,τοῦ,3588,21,22,66,t-gsm
137521,κυρίου,2962,21,22,66,n-gsm
137522,ἰησοῦ,2424,21,22,66,n-gsm
137523,μετὰ,3326,21,22,66,prep


In [4]:
sept_verses = list(
    sept.groupby(["book", "chapter", "verse"])["text"]
    .apply(lambda words: " ".join(words))
    .values
)
tisch_verses = list(
    tisch.groupby(["book", "chapter", "verse"])["text"]
    .apply(lambda words: " ".join(words))
    .values
)

In [5]:
tisch_verse_df = pd.DataFrame(data=tisch_verses, columns=["verse"])
sept_verse_df = pd.DataFrame(data=sept_verses, columns=["verse"])
display(tisch_verse_df.head(3))

,verse
0,βίβλος γενέσεως ἰησοῦ χριστοῦ υἱοῦ δαυεὶδ υἱοῦ...
1,ἀβραὰμ ἐγέννησεν τὸν ἰσαάκ ἰσαὰκ δὲ ἐγέννησεν ...
2,ἰούδας δὲ ἐγέννησεν τὸν φάρες καὶ τὸν ζάρα ἐκ ...


In [6]:
tisch_verse_df.loc[0]

verse    βίβλος γενέσεως ἰησοῦ χριστοῦ υἱοῦ δαυεὶδ υἱοῦ...
Name: 0, dtype: object

In [7]:
def word_vectorise(sentence: str) -> np.array:
    return [wordvec_model.wv[word] for word in sentence.split()]


tisch_verse_df["word_vec"] = tisch_verse_df["verse"].apply(word_vectorise)
sept_verse_df["word_vec"] = sept_verse_df["verse"].apply(word_vectorise)
print("length:", len(tisch_verse_df.loc[0, "word_vec"][0]))
print(tisch_verse_df.loc[0, "word_vec"][0])

length: 100
[-2.15206340e-01  8.58220235e-02  2.31477898e-02  2.91070670e-01
  6.88552931e-02 -1.29276782e-01  4.40058857e-02  1.08505890e-01
 -9.32898074e-02 -1.88845873e-01  9.34656039e-02 -1.71858899e-03
 -2.59595662e-01 -1.37818202e-01  7.24177957e-02 -1.45290837e-01
  8.81084725e-02 -2.21661896e-01 -4.60098311e-02 -2.89330035e-01
  1.05094522e-01  1.37629911e-01  2.63089508e-01 -1.00925855e-01
  1.78266689e-02 -7.71997720e-02 -1.00133605e-01 -1.21119395e-01
 -9.42055434e-02  8.01312253e-02 -6.25229552e-02 -1.97353467e-01
  1.59132794e-01 -2.03227520e-01 -1.42959496e-02  1.72883660e-01
 -1.01664551e-02 -7.27054924e-02 -3.99184749e-02 -2.75744885e-01
  1.05004594e-01 -1.65659338e-01  9.42698941e-02  5.55301979e-02
 -1.62983894e-01 -9.46601853e-03 -1.26292884e-01  7.67456144e-02
  5.16312644e-02  8.57987383e-04  2.86514282e-01 -8.10800195e-02
  5.88460863e-02 -4.47198786e-02 -2.46748270e-04  3.71960178e-02
 -1.79488331e-01  9.92837623e-02 -6.87277317e-02  1.39979094e-01
 -1.16153762e

In [17]:
sept_verse_vecs = sept_verse_df["word_vec"].values
tisch_verse_vecs = tisch_verse_df["word_vec"].values

In [18]:
with open("pickles/sept_verse_vecs.pickle", "wb") as f:
    pickle.dump(sept_verse_vecs, f)
with open("pickles/tisch_verse_vecs.pickle", "wb") as f:
    pickle.dump(tisch_verse_vecs, f)